# ISOM 839 · Session 9 — Markov Chains: Modeling Customer Journeys

**Prescriptive Analytics: Modeling & Optimization · Suffolk University · Prof. Hasan Arslan**

Every customer is somewhere in a journey — browsing, buying, lapsing, returning. Tonight we model those journeys as **Markov chains**: a transition matrix, a population vector, and two lines of numpy that turn churn into a dollar figure a CFO will accept. The same math, with web pages for states, built Google.

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
np.set_printoptions(precision=3, suppress=True)

---
## Part 1 — The journey as a matrix (tonight's lab)

A $15/month subscription business. Four states: **Active, At-Risk, Lapsed, Churned** (absorbing — nobody comes back from Churned). One row per state; each row is *where you go next* and must sum to 1. That row-sum check is the most common Markov bug in the wild, so we test it first.

In [ ]:
states = ['Active', 'At-Risk', 'Lapsed', 'Churned']
P = np.array([
    [0.80, 0.15, 0.03, 0.02],   # Active
    [0.30, 0.40, 0.20, 0.10],   # At-Risk
    [0.10, 0.10, 0.50, 0.30],   # Lapsed
    [0.00, 0.00, 0.00, 1.00],   # Churned (absorbing)
])
assert np.allclose(P.sum(axis=1), 1), 'every row must sum to 1'
print(pd.DataFrame(P, index=states, columns=states))

**Step 2 — project the cohort.** 10,000 new customers start Active. Multiply by P once per month.

In [ ]:
x = np.array([10_000.0, 0, 0, 0])
history = [x]
for month in range(36):
    x = x @ P
    history.append(x)
H = np.array(history)
for m in (12, 24, 36):
    print(f'month {m:2d}:', {s: int(round(v)) for s, v in zip(states, H[m])})
plt.figure(figsize=(7, 3.6)); plt.stackplot(range(37), H.T, labels=states, alpha=.85); plt.legend(loc='upper right'); plt.xlabel('month'); plt.ylabel('customers'); plt.show()

**Step 3 — the steady state.** Keep multiplying and the distribution stops changing. With an absorbing state, that limit is boring — everyone churns eventually — so the useful question is *how long it takes*, which is the transient behavior you just plotted.

In [ ]:
x = np.array([10_000.0, 0, 0, 0])
for month in range(1, 2000):
    x_new = x @ P
    if np.max(np.abs(x_new - x)) < 1e-6:
        break
    x = x_new
print(f'converged after {month} months:', {s: round(float(v), 1) for s, v in zip(states, x)})

**Step 4 — customer lifetime value.** Split P into the transient block **Q** (the 3×3 corner) and compute the fundamental matrix **N = (I − Q)⁻¹**. Entry N[i, j] is the expected number of months a customer starting in state i spends in state j before being absorbed. Multiply by what each state pays, and you have CLV.

In [ ]:
Q = P[:3, :3]
N = np.linalg.inv(np.eye(3) - Q)
margin = np.array([15.0, 15.0, 0.0])        # $/month: Lapsed customers pay nothing
clv = N @ margin
print('expected months in each state, starting Active:', {s: round(float(v), 2) for s, v in zip(states[:3], N[0])})
print(f'expected lifetime: {N[0].sum():.1f} months   CLV of a new Active customer: ${clv[0]:,.2f}')

def clv_of(P):
    Q = P[:3, :3]; N = np.linalg.inv(np.eye(3) - Q); return (N @ margin)[0]

P_a = P.copy(); P_a[0, 0] += 0.05; P_a[0, 1] -= 0.05      # Active retention 0.80 -> 0.85
P_b = P.copy(); P_b[1, 0] += 0.10; P_b[1, 1] -= 0.10      # At-Risk recovery 0.30 -> 0.40
print(f'\nLift Active retention to 0.85:  CLV ${clv_of(P_a):,.2f}  (+${clv_of(P_a) - clv[0]:,.2f})')
print(f'Lift At-Risk recovery to 0.40:  CLV ${clv_of(P_b):,.2f}  (+${clv_of(P_b) - clv[0]:,.2f})')

### 💬 Discussion
Your model says a new customer is worth a specific dollar amount — so what is the *maximum* you would pay to acquire one? The two interventions each cost $2 per customer per month. Which one wins, and is it the one your marketing team would have guessed? (Small lifts on the state where people *spend the most months* beat big lifts on states they pass through quickly.)

### PageRank, in one sentence
Replace customers with web pages and transitions with links: the steady state of that chain is Google's original ranking. Same `x @ P`, forty billion rows.

---
# Homework #6 — Churn Markov Model (Streaming Subscribers) 📺

**On Canvas: this notebook completed + your go / no-go paragraph in the text box.** Due date and rubric on the Canvas assignment.

A streaming service tracked a cohort of **5,000 new subscribers** for twelve months. Each month every subscriber is in one of five states: **New** (first month only), **Engaged**, **Casual**, **Dormant** (still subscribed, not watching) and **Cancelled** (absorbing). Every subscribed state pays a **$12.99 monthly margin**. The file has one row per subscriber and one column per month.

In [ ]:
cohort = pd.read_csv('https://raw.githubusercontent.com/harslan/isom-839/master/public/session-09/streaming_cohort.csv')
states = ['New', 'Engaged', 'Casual', 'Dormant', 'Cancelled']
months = [c for c in cohort.columns if c.startswith('month_')]
print(cohort.head()); print('\nsubscribers:', len(cohort), '· months observed:', len(months) - 1)
print('\nstate mix at month 12:'); print(cohort['month_12'].value_counts().reindex(states).fillna(0).astype(int))

1. **Estimate the transition matrix** from the data: count every month-to-month move (subscriber i, month t → t+1), divide each row by its total. Show the 5×5 matrix and check the rows sum to one. Cancelled is absorbing — set that row by hand if your counts leave it empty.
2. **Project the subscriber base 24 months forward** from 5,000 New, and plot the population of each state over time.
3. **Expected lifetime value per new subscriber** at the $12.99 margin, via the fundamental matrix. Show the calculation, not just the number.
4. **The win-back campaign.** Management proposes moving the Dormant → Engaged probability from 0.05 to 0.15 (taking the 0.10 out of Dormant → Dormant), at a cost of **$3 per Dormant subscriber per month**. Recommend **go / no-go** with numbers: the change in lifetime value against the expected campaign cost per new subscriber, and the one assumption that could flip your answer.

In [ ]:
# TODO 1: count transitions and estimate P (5x5); assert rows sum to 1

# TODO 2: project 5,000 New subscribers 24 months forward; plot the state populations

# TODO 3: fundamental matrix N = (I - Q)^-1 over the four transient states; LTV = 12.99 * expected months subscribed

# TODO 4: modify the Dormant row, recompute LTV; campaign cost = 3 * expected months in Dormant; go / no-go

---
### Submission checklist
- [ ] Estimated matrix printed; rows sum to one; Cancelled row is absorbing
- [ ] 24-month projection plotted
- [ ] LTV shown as a calculation (N, then dollars)
- [ ] Go / no-go paragraph with the value, the cost, and the assumption that could flip it — pasted into the Canvas text box

**Next class — Session 10 on November 18** (no class November 11, Veterans Day): chains learn to choose. States, actions, rewards, and the Bellman equation. 🎯